# Data Cleaning 

This notebook cleans the raw data available in data/raw and writes the clean version back to the folder data/processed. 

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from c08_farming_exit import config, features, data_cleaning, mappings

In [54]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

In [15]:
# missing_by_country = (
#     df_database
#     .groupby('country')
#     .apply(lambda g: g.isna().mean())
#     .sort_index()
# )
# missing_by_country

In [2]:
COUNTRIES = {
    "Botswana": config.RAW_DATA_DIR / "Botswana",
    "Kenya":    config.RAW_DATA_DIR / "Kenya",
    "Namibia":  config.RAW_DATA_DIR / "Namibia",
    "Tanzania": config.RAW_DATA_DIR / "Tanzania",
    "Zambia":   config.RAW_DATA_DIR / "Zambia",
}

## 1. Database

In [7]:
#THE DATABASE CONSISTS OF ADULTS ONLY
database = []

for country, base_path in COUNTRIES.items():
    identifying_info    = data_cleaning.load_and_preprocess(base_path, f"{country}_identifying_info.csv",                  features.IDENTIFYING_INFO_2023)
    
    hh_members          = data_cleaning.load_and_preprocess(base_path, f"{country}_household_members_characterstics.csv",  features.HH_MEMBERS_2023)
    hh_members          = data_cleaning.add_years_of_schooling(hh_members, mappings.education_mapping)

    merge = identifying_info.merge(hh_members, on=["interview_key"], how="inner")
    
    filtered = merge[
        merge["relation_to_head"].isin([    "Self/Head", 
                                            "Wife/Husband", 
                                            "Son/Daughter-In-Law", 
                                            "Sister/Brother", 
                                            "Mother/Father", 
                                            "Brother/Sister-In-Law", 
                                            "Grandfather/Mother", 
                                            "Father/Mother-In-Law"]) &
                                        (merge["age"] >= 18)
    ]

    database.append(filtered)

df_database = pd.concat(database, ignore_index=True)

#CREATE PERSONAL IDENTIFIER 
df_database["personal_id"] = df_database["country"] + "_" + df_database["interview_key"].astype(str) + "_" + df_database["members_id"].astype(str)

#FINAL SORTING
df_database = df_database[['country', 'region', 'district', 'enumeration_area', 'personal_id', 'interview_key', 'relation_to_head', 'gender', 'age', 'years_of_schooling', 'education_level']] \
                .query("country != 'YES+A112:L126+A112:C126'")

[Botswana_identifying_info.csv] Missing columns: ['ea', 'region']
[Namibia_identifying_info.csv] Missing columns: ['dist']


## 2. Features

### 2.1 Creating HH-Level Features

In [ ]:
hh_features = []

for country, base_path in COUNTRIES.items():
    #NO CLEANING NECESSARY - one observation per hh
    land_ownership          = data_cleaning.load_and_preprocess(base_path, f"{country}_land_ownership_and_access.csv",         features.LAND_OWNERSHIP_ACCESS_2023)
    land_ownership          = data_cleaning.convert_land_sizes_to_acres(land_ownership, country)

    crop_expenditure        = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_on_crops.csv",              features.CROP_EXPENDITURE_2023)

    # lifestock_grazing       = data_cleaning.load_and_preprocess(base_path, f"{country}_grazing_patterns_and_schemes.csv",      features.LIFESTOCK_GRAZING_2023)
    # livestock_income        = data_cleaning.load_and_preprocess(base_path, f"{country}_income_livestock.csv",                  features.LIFESTOCK_INCOME_2023)
    # livestock_expenditure   = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_livestock.csv",             features.LIFESTOCK_EXPENDITURE_2023)
    # housing_conditions      = data_cleaning.load_and_preprocess(base_path, f"{country}_housing_conditions.csv",                features.HOUSING_CONDITIONS_2023)
    # energy_access           = data_cleaning.load_and_preprocess(base_path, f"{country}_access_to_energy.csv",                  features.ENERGY_ACCESS_2023)
    # internet_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_internet_access.csv",                   features.INTERNET_ACCESS_2023)
    # social_network          = data_cleaning.load_and_preprocess(base_path, f"{country}_other_household_social_network.csv",    features.SOCIAL_NETWORK_2023)
    # social_embeddedness     = data_cleaning.load_and_preprocess(base_path, f"{country}_social_embeddedness.csv",               features.SOCIAL_EMBEDDEDNESS_2023)
    # food_insecurity         = data_cleaning.load_and_preprocess(base_path, f"{country}_food_insecurity_experiance_scale.csv",  features.FOOD_INSECURITY_2023)
    # road_connectivity       = data_cleaning.load_and_preprocess(base_path, f"{country}_road_connectivity.csv",                 features.ROAD_CONNECTIVITY_2023)

    # #CLEANING NECESSARY - many observations per hh
    # market_access           = data_cleaning.load_and_preprocess(base_path, f"{country}_market_access.csv",                     features.MARKET_ACCESS_2023)
    # crop_production         = data_cleaning.load_and_preprocess(base_path, f"{country}_crop_production.csv",                   features.CROP_PRODUCTION_2023) #different crops
    # livestock_ownership     = data_cleaning.load_and_preprocess(base_path, f"{country}_livestock_ownership.csv",               features.LIVESTOCK_OWNERSHIP_2023) #different animals
    # assets_owned            = data_cleaning.load_and_preprocess(base_path, f"{country}_assets.csv",                            features.ASSETS_OWNED_2023)
    # shocks_and_coping       = data_cleaning.load_and_preprocess(base_path, f"{country}_shocks_and_coping.csv",                 features.SHOCKS_AND_COPING_2023)
    # other_income            = data_cleaning.load_and_preprocess(base_path, f"{country}_other_income.csv",                      features.OTHER_INCOME_SOURCES_2023)


    #SOME TABLES ARE NOT AVAILABLE FOR EACH COUNTRY: optional_merges solves this as it only merges available tables
    optional_merges = [
        (land_ownership,     ["interview_key"],     "outer"),
        (crop_expenditure,   ["interview_key"],     "outer"),
        # (crop_production,    ["interview_key"],     "outer")
    ]

    df_help = None

    for df, keys, how in optional_merges:
        if df is not None:
            if df_help is None:
                df_help = df
            else:
                df_help = df_help.merge(df, on=keys, how=how)
                
    #Adding the country to the table for identification
    df_help.insert(0, "country", country)

    hh_features.append(df_help)


df_hh_features = pd.concat(hh_features, ignore_index=True)



convert_land_sizes_to_acres: Tanzania - Dropped 1 rows with NaN in 'land_measurement'


### 2.2 Creating Individual-Level Features

In [62]:
individual_features = []

for country, base_path in COUNTRIES.items():
    #NO CLEANING NECESSARY 

    #CLEANING NECESSARY


    #SOME FEATURES ARE NOT AVAILABLE FOR EACH COUNTRY: optional_merges solves this as it only merges available features
    optional_merges = [
    ]

    df_help = None

    for df, keys, how in optional_merges:
        if df is not None:
            if df_help is None:
                df_help = df
            else:
                df_help = df_help.merge(df, on=keys, how=how)

    individual_features.append(df_help)

df_individual_features = pd.concat(individual_features, ignore_index=True)


ValueError: All objects passed were None

## 3. Write clean data to data/raw folder

In [ ]:
df.to_csv(config.PROCESSED_DATA_DIR / "clean_data.csv", index=False)